In [279]:
import pandas as pd
import numpy as np

In [280]:
def generate_features(n_samples, n_features, prefix):
    df = pd.DataFrame()
    
    nominal = [1000, 500, 333, 777]
    ordinal = [1, 2, 3, 4]
    
    for i in range(1, n_features + 1):
        if i % 4 == 1:
            df[f'{prefix}_bin_{i}'] = np.random.choice([1, 0], n_samples)
        elif i % 4 == 2:
            df[f'{prefix}_nominal_{i}'] = np.random.choice(nominal, n_samples)
        elif i % 4 == 3:
            df[f'{prefix}_ordinal_{i}'] = np.random.choice(ordinal, n_samples)
        else:
            df[f'{prefix}_quantitative_{i}'] = np.random.uniform(10, 1000, n_samples)
            
    return df


In [281]:
df1 = generate_features(10, 3, "test")
df2 = generate_features(10, 3, "test")

In [282]:
df1

,test_bin_1,test_nominal_2,test_ordinal_3
0,1,1000,1
1,0,333,2
2,0,500,4
3,0,777,3
4,1,333,2
5,0,777,2
6,0,333,2
7,0,777,1
8,0,1000,4
9,0,1000,3


In [283]:
df1.iloc[1]

test_bin_1          0
test_nominal_2    333
test_ordinal_3      2
Name: 1, dtype: int64

In [284]:
def collision(row1, row2):
    diff_sum = 0
    v1 = row1.values
    v2 = row2.values
    cols = row1.index 
    
    for i in range(len(cols)):
        col_name = cols[i]
        val1 = v1[i]
        val2 = v2[i]
        
        if 'bin' in col_name or 'nominal' in col_name:
            diff_sum += 1 if val1 != val2 else 0
        elif 'ordinal' in col_name:
            if val1 == 777 or val2 == 777:
                diff_sum += 2
            elif val1 == 333 or val2 == 333:
                diff_sum += 1
        else:
            diff_sum += abs(val1 - val2) / 1000
    
    return 1 if diff_sum < len(cols) * 0.4 else 0

In [285]:
def create_dataset(n_samples, n_features_obj1, n_features_obj2):
    obj1_df = generate_features(n_samples, n_features_obj1, 'Obj1')
    obj2_df = generate_features(n_samples, n_features_obj2, 'Obj2')
    
    dataset = pd.concat([obj1_df, obj2_df], axis=1)
    
    results = []
    for i in range(len(dataset)):
        res = collision(obj1_df.iloc[i], obj2_df.iloc[i])
        results.append(res)

    dataset['collision'] = results
    
    return dataset


In [286]:
create_dataset(10, 4, 4)

,Obj1_bin_1,Obj1_nominal_2,Obj1_ordinal_3,Obj1_quantitative_4,Obj2_bin_1,Obj2_nominal_2,Obj2_ordinal_3,Obj2_quantitative_4,collision
0,1,500,1,337.317729,0,1000,2,225.891603,0
1,1,333,3,274.510763,1,500,1,130.276133,1
2,1,1000,1,796.110494,1,1000,1,464.204161,1
3,0,500,2,412.785969,1,500,1,586.191787,1
4,0,333,4,613.309632,0,777,1,502.855589,1
5,1,333,2,343.989754,0,1000,4,283.134847,0
6,0,333,3,539.826677,1,777,4,424.326327,0
7,1,333,1,478.743786,0,333,1,586.157717,1
8,1,333,2,631.629050,0,333,3,351.940653,1
9,0,500,4,623.481431,0,333,2,928.406104,1


In [287]:
sample_sizes = [50, 250, 750, 1500]
feature_counts = [5, 9, 15]
datasets = {}
dataset_id = 1

for n_samples in sample_sizes:
    for n_features in feature_counts:
        df = create_dataset(n_samples, n_features, n_features)
        name = f"ID:{dataset_id}_S{n_samples}_F{n_features}"
        datasets[name] = df
        dataset_id += 1
        df.to_csv(name)

for ds in datasets:
    print(ds)

datasets["ID:3_S50_F15"].head()

ID:1_S50_F5
ID:2_S50_F9
ID:3_S50_F15
ID:4_S250_F5
ID:5_S250_F9
ID:6_S250_F15
ID:7_S750_F5
ID:8_S750_F9
ID:9_S750_F15
ID:10_S1500_F5
ID:11_S1500_F9
ID:12_S1500_F15


,Obj1_bin_1,Obj1_nominal_2,Obj1_ordinal_3,Obj1_quantitative_4,Obj1_bin_5,Obj1_nominal_6,Obj1_ordinal_7,Obj1_quantitative_8,Obj1_bin_9,Obj1_nominal_10,...,Obj2_ordinal_7,Obj2_quantitative_8,Obj2_bin_9,Obj2_nominal_10,Obj2_ordinal_11,Obj2_quantitative_12,Obj2_bin_13,Obj2_nominal_14,Obj2_ordinal_15,collision
0,0,500,1,803.450495,0,500,3,949.407036,1,500,...,1,808.691849,1,333,4,832.399412,1,1000,4,0
1,1,333,4,878.476718,1,777,4,565.813196,0,777,...,4,175.470366,1,1000,1,196.776557,0,1000,4,0
2,0,777,3,872.058186,1,777,3,525.622970,1,1000,...,1,443.344426,1,333,3,200.285524,0,333,2,0
3,0,500,3,170.443486,1,1000,1,190.734275,1,777,...,3,170.223426,0,333,4,535.599171,1,777,1,0
4,1,500,3,289.059593,0,333,2,799.457669,1,333,...,2,506.100336,1,777,2,450.512527,1,777,4,1
